# Load Libraries and Packages

In [ ]:
# Load libraries
import os
import pandas as pd
import re
from openai import OpenAI
# OpenAI API key
openai_key = "your key"
client = OpenAI(api_key=openai_key)


In [ ]:
# Load the testing set data
test_df = pd.read_csv('test_data_forGPT.csv')
test_df


# Choosing a GPT model


In [ ]:
from tqdm import tqdm  # Import tqdm for progress bar
import time
tqdm.pandas()  # Enable tqdm progress bar

# now I want to use gpt4o as the model for zero-shot classification
from typing import List, Dict
DEFAULT_TEMPERATURE = 0
DEFAULT_MODEL = "gpt-4o" #"gpt-3.5-turbo-0125" 
# Define a function to prompt via GPT
def send_prompt_with_context(messages: List[Dict], model: str = DEFAULT_MODEL) -> str:
    max_retries = 5  # Retry up to 5 times
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=DEFAULT_TEMPERATURE
            )
            answer = resp.choices[0].message.content
            return answer
        except Exception as e:
            print(f"Error in GPT request (Attempt {attempt+1}/{max_retries}): {e}")
            time.sleep(3)  # Small delay before retrying

    return "{}"  # Return empty JSON object if all retries fail


# Three ways: Zero-shot, Few-shot learning, and Finetuned GPT, can choose one of them depending on your need.

## Option 1: Zero-Shot Prompt for Perceptions of Parks

In [ ]:
def predict_perception_parks(review: str) -> Dict[str, int]:
    system_msg = """
    You are a helpful urban researcher who classifies perceptions of parks using users' reviews. Respond in a structured format with one key: attractiveness.
    Attractiveness refers to how well a park aligns with individual preferences, influencing both the likelihood of voluntary visits and the duration of stay. Key contributing factors may include park size, aesthetic quality, natural features, noise levels, and use of commercial facilities.  "
    Labeling Rules: Attractiveness: Label as 1 if positive, 0 if negative, and 2 if irrelevant. You will only return 1 or 0 or 2 for each Label.
    Respond with only valid JSON.
    """    
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": f"Review: {review}"}
    ]
    return send_prompt_with_context(messages)   

## Option 2: Few-Shot Prompt for Perceptions of Parks

In [ ]:
# please note we added 20 representative labels as examples (in the following code we only put 4 for illustrative purpose)

def predict_perception_parks(review: str) -> Dict[str, int]:
    system_msg = """
    You are a helpful urban researcher who classifies perceptions of parks using users' reviews. Respond in a structured format with one key: attractiveness.
    Attractiveness refers to how well a park aligns with individual preferences, influencing both the likelihood of voluntary visits and the duration of stay. Key contributing factors may include park size, aesthetic quality, natural features, noise levels, and use of commercial facilities.  "
    Labeling Rules: Attractiveness: Label as 1 if positive, 0 if negative, and 2 if irrelevant. You will only return 1 or 0 or 2 for each Label.
    
    ### Examples:
    Review: "A precious oasis on Hong Kong Island where you can spend as much time as you want."
    Response: {'attractiveness': 1}

    Review: "There is an elevator next to Choi Fun House in Choi Fook Estate that can be accessed."
    Response: {'attractiveness': 2}

    Review: "The playground facilities are very poor; the park is so big, but the playground is very small and not fun at all."
    Response: {‘attractiveness': 0}

    Review: "Transportation is very convenient! Along the way, you can enjoy the views of both sides of Victoria Harbour. It's a great place for walking and taking photos."
    Response: {attractiveness': 1}
    

    
    Respond with only valid JSON.
    """
    
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": f"Review: {review}"}
    ]
    return send_prompt_with_context(messages)   

## Option 3: finetuned GPT model inferral

In our research we further conducted Fine-tuning, which can be conducted in the OpenAI platform to fine-tune a customized model with an unique model ID.

The input for finetuning is a jsonl file containing your prompt + comments + labels (using all your training data)

In [ ]:
from typing import List, Dict
DEFAULT_TEMPERATURE = 0
DEFAULT_MODEL = "ft:gpt-4o-mini-2024-07-18:personal:your customized model id" # this is the finetuned model from GPT, here I use GPT-4o-mini
def send_prompt_with_context(messages: List[Dict], model: str = DEFAULT_MODEL) -> str:
    max_retries = 5  # Retry up to 5 times
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=DEFAULT_TEMPERATURE
            )
            answer = resp.choices[0].message.content
            return answer
        except Exception as e:
            print(f"Error in GPT request (Attempt {attempt+1}/{max_retries}): {e}")
            time.sleep(3)  # Small delay before retrying

    return "{}"  # Return empty JSON object if all retries fail


In [ ]:
# here we are prompting using the 'fine-tuned model'
def predict_perception_parks(review: str) -> Dict[str, int]:
    system_msg = """
    You are a helpful urban researcher who classifies perceptions of parks using users' reviews. Respond in a structured format with one key: attractiveness.
    Attractiveness refers to how well a park aligns with individual preferences, influencing both the likelihood of voluntary visits and the duration of stay. Key contributing factors may include park size, aesthetic quality, natural features, noise levels, and use of commercial facilities.  "
    Labeling Rules: Attractiveness: Label as 1 if positive, 0 if negative, and 2 if irrelevant. You will only return 1 or 0 or 2 for each Label.
    Respond with only valid JSON.
    """    
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": f"Review: {review}"}
    ]
    return send_prompt_with_context(messages)   

# Prediction Perception

In [ ]:
# Apply function with tqdm progress bar
test_df.loc[:, "predictions"] = test_df["review_texts"].progress_apply(predict_perception_parks)

In [ ]:
# Define a function to extract the number from a string
def extract_numbers(text):
    numbers = re.findall(r'\d+', str(text))  # Find all numbers
    numbers = [int(num) for num in numbers[:1]]  # Convert to integers and take only the first three
    while len(numbers) < 1:
        numbers.append(None)  # Fill missing values with None
    return numbers

# Apply function to extract numbers
test_df[['gpt_attractiveness_4o']] = test_df['predictions'].apply(lambda x: pd.Series(extract_numbers(x)))

test_df

In [ ]:
# export to an excel sheet
test_df.to_excel('1 gpt4o_results.xlsx', index = False)